# **Statistical Learning for Automation Systems**

       
## *Prof. Simone Formentin*

___

# Regression with Neural Networks - Case Study

## Problem Statement

Let's consider a dataset containing measurements related to some rare gem stones, with 2 measured features and the sale price. Our final goal would be to try to predict the sale price of a new gem stone we just mined from the ground, in order to try to set a fair price in the market.

## Import Data Analysis and Visualization Libraries

In [ ]:
%pip install numpy pandas matplotlib seaborn openpyxl
%pip install ipywidgets

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

%matplotlib inline

## Import Dataset

In [ ]:
df = pd.read_csv('gem_stones.csv')            # Import Dataset from .csv
df.head()                                     # Show header

In [ ]:
df.info()

In [ ]:
df.describe()

## Exploratory Data Analysis

In [ ]:
sns.pairplot(df)

An approximated linear relationship is fairly evident between feature2 and price.

## Test/Train Set Split

In [ ]:
from sklearn.model_selection import train_test_split        # Import train_test_split function from Scikit-Learn

# Features (convert from DataFrame to NumPy array)
X = np.array(df[['feature1','feature2']])

# Target Variable
y = np.array(df['price'])

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

## Data Pre-Processing

Use StandardScaler() to perform input centering and input normalization.

Notice that it is sufficient to standardize just input features, while it is not necessary to standardize the target variable.

In [ ]:
sns.displot(X_train[:,0], kde = True)

### Standard scaler: $Z=\frac{X-\mu}{\sigma}$

Notice: to prevent data leakage (use information from data outside the training set) from the test set, we only fit our scaler on the training set.

In [ ]:
from sklearn.preprocessing import StandardScaler

# StandardScaler() object instantiation
scaler = StandardScaler()

# Fit Standard Scaler only on the training set
scaler.fit(X_train)

# Apply standardization
X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)

#### Showing importance of scaling

In [ ]:
from sklearn.linear_model import SGDRegressor
from sklearn.metrics import mean_squared_error

# Synthetic data
np.random.seed(0)
engine_size = np.random.uniform(1, 6, 100)
weight = np.random.uniform(800, 3000, 100)
y_example = 40 - 3 * engine_size - 0.005 * weight + np.random.randn(100) * 2

# Case 1: Without normalization
X_raw = np.column_stack([engine_size, weight])
model_raw = SGDRegressor(max_iter=1000, tol=1e-3).fit(X_raw, y_example)
print("Without normalization MSE:", mean_squared_error(y_example, model_raw.predict(X_raw)))

# Case 2: With normalization
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)
model_scaled = SGDRegressor(max_iter=1000, tol=1e-3).fit(X_scaled, y_example)
print("With normalization MSE:", mean_squared_error(y_example, model_scaled.predict(X_scaled)))

## Neural Network Definition - TensorFlow & Keras

There are several ways you can import Keras from Tensorflow. We will use the method shown in the **official TF documentation**.

## Creating a NN Model

In [ ]:
from tensorflow.keras.models import Sequential               # Import Sequential() to instatiate a Feed-forward NN object
from tensorflow.keras.layers import Dense                    # Import Dense() to instatiate fully-connected layer objects

### NN Model - Adding layers one by one

**Sequential model**

- When to use it:
    A Sequential model is appropriate for a plain stack of layers where each layer has exactly one input tensor and one output tensor.
- When **not** to use it:
    - Your model has multiple inputs or multiple outputs
    - Any of your layers has multiple inputs or multiple outputs
    - You need to do layer sharing
    - You want non-linear topology (e.g. a residual connection, a multi-branch model)
- How to use it: You can create a Sequential model by passing a list of layers to the Sequential constructor



In [ ]:
model = Sequential()

model.add(Dense(4, activation='relu'))
model.add(Dense(4, activation='relu'))
model.add(Dense(4, activation='relu'))

# Final output node for prediction
model.add(Dense(1))                           # In the output layer we just have one neuron since the target is a single variable (price)

model.compile(optimizer='rmsprop', loss='mse')      # Optimizer determines the way in which you approach gradient descent (alternative: ADAM)

Optimizer:
The optimizer adjusts the network’s weights to minimize the loss function (e.g., Mean Squared Error in regression). It determines how the model learns from the data by controlling the step size and direction of each update.

Examples:

 - SGD (Stochastic Gradient Descent): updates weights using gradients from small batches; simple but can be slow.

 - Adam: combines momentum and adaptive learning rates, offering faster and more stable convergence — widely used in regression and general tasks.

 - RMSprop: adapts the learning rate based on recent gradient magnitudes, effective for noisy or non-stationary data.

### Choosing suitable Loss functions and Evaluation Metrics

**Gradient Descent:**  
Gradient descent is an optimization algorithm used to minimize a loss function by iteratively updating the model’s parameters in the direction of the *negative gradient*. The idea is to move step by step toward the point where the loss is smallest.  

At each iteration, the weights $ w $ are updated as:  

$$
w \leftarrow w - \eta \nabla_w L(w)
$$

where $\eta$ is the learning rate (step size) and $\nabla_w L(w) $ is the gradient of the loss with respect to the weights.  

In essence, gradient descent helps the neural network learn by gradually adjusting its parameters to reduce prediction error.




Keep in mind what kind of problem you are trying to solve:

    # For a multi-class classification problem
    model.compile(optimizer='rmsprop',
                  loss='categorical_crossentropy',
                  metrics='accuracy')

    # For a binary classification problem
    model.compile(optimizer='rmsprop',
                  loss='binary_crossentropy',
                  metrics='accuracy')

    # For a mean squared error regression problem
    model.compile(optimizer='rmsprop',
                  loss='mse')

## Training NN Model


In [ ]:
model.fit(X_train, y_train, epochs=250)      

## Evaluation

Let's evaluate our performance on our training set and our test set. We can compare these two performances to check for overfitting.

In [ ]:
# Check available metrics
print(model.metrics_names)

# Store performance over time
loss = model.history.history['loss']

# Plot Training Loss per Epoch
plt.figure(figsize=(6,3))
sns.lineplot(x=range(len(loss)),y=loss)
plt.title("Training Loss per Epoch")
plt.xlabel('Epoch')
plt.ylabel('Training Loss Function (MSE)')
plt.grid()
# plt.yscale('log')

## Loss evaluation on Training set and Test set.

These should hopefully be fairly close to each other.

In [ ]:
# Evaluate the performance according to a metric (here, the loss function)
training_score = model.evaluate(X_train, y_train, verbose=0)
test_score = model.evaluate(X_test, y_test, verbose=0)

# Print results
print('Training score:', training_score)
print('Test score:', test_score)

### Further Evaluations

In [ ]:
# Generate the predictions and the ground truth
test_predictions = model.predict(X_test)
pred_df = pd.DataFrame(y_test, columns=['Test Y'])

# Generate a single DataFrame for both outputs
test_predictions = pd.Series(test_predictions.reshape(300,))
pred_df = pd.concat([pred_df, test_predictions],axis=1)
pred_df.columns = ['Test Y','Model Predictions']

# Display result
pred_df.head()

## Real Test Set Labels vs NN Model Predictions

In [ ]:
plt.figure(figsize = (5,5))
sns.scatterplot(x='Test Y', y='Model Predictions', data=pred_df)
plt.title('Real Test Labels vs NN Model Predictions')
plt.grid()

In [ ]:
# Display prediction error
pred_df['Error'] = pred_df['Test Y'] - pred_df['Model Predictions']
sns.displot(pred_df['Error'], bins=50, kde=True)

In [ ]:
# Evaluate metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error
print('MAE:', mean_absolute_error(pred_df['Test Y'], pred_df['Model Predictions']))         # Prediction Error MAE
print('MSE', mean_squared_error(pred_df['Test Y'], pred_df['Model Predictions']))           # Prediction Error MSE
print('Test score:', test_score)
print('RMSE:', test_score**0.5)

# Predicting on brand new data

What if we just saw a brand new gemstone from the ground? What should we price it at? This is the **exact** same procedure as predicting on a new test data!

In [ ]:
# [[Feature1, Feature2]]
new_gem = [[998, 1000]]

# Don't forget to standardize the new input!
new_gem = scaler.transform(new_gem)

# Predict
model.predict(new_gem)

# Validation set
**Useful best practice** use a validation loss and monitor it because it's a proxy for overfitting

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.1, random_state=42)
X_train.shape, X_val.shape, X_test.shape

In [ ]:
# Standardize the data
scaler = StandardScaler()
scaler.fit(X_train)
X_train = scaler.transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)
X_train.shape, X_val.shape, X_test.shape

In [ ]:
model = Sequential()

model.add(Dense(4, activation='relu'))
model.add(Dense(4, activation='relu'))
model.add(Dense(4 ,activation='relu'))

# Final output node for prediction
model.add(Dense(1))                           # In the output layer we just have one neuron since the target is a single variable (price)

model.compile(optimizer='rmsprop', loss='mse') 
model.fit(X_train, y_train, epochs=250, validation_data=(X_val, y_val))

In [ ]:
# Plot Training and Validation Loss per Epoch
loss = model.history.history['loss']
val_loss = model.history.history['val_loss']
plt.figure(figsize=(6,3))
sns.lineplot(x=range(len(loss)), y=loss, label='Training Loss')
sns.lineplot(x=range(len(val_loss)), y=val_loss, label='Validation Loss')
plt.title("Training and Validation Loss per Epoch")
plt.xlabel('Epoch')
plt.ylabel('Loss Function (MSE)')
plt.legend()
plt.yscale('log')
plt.show()

# Saving and Loading a Model

In [ ]:
from tensorflow.keras.models import load_model

# Save model (creates a HDF5 file 'my_model.h5')
model.save('my_model.keras')

# Load model
saved_model = load_model('my_model.keras')

# Predict with loaded model
saved_model.predict(new_gem)